# 1. Installing and importing libraries

In [ ]:
!pip install uv

In [ ]:
!uv pip install numpy pandas torch optuna sdmetrics ctgan gdown xlrd polars fastexcel

In [ ]:
import os
import gc
import json
import time
import shutil
import resource
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
import torch

from scipy import stats as sstats
from scipy.stats import wasserstein_distance, ks_2samp, anderson_ksamp
from scipy.spatial.distance import jensenshannon, cdist

from sklearn.preprocessing import StandardScaler

import optuna
from optuna.samplers import TPESampler

from ctgan import CTGAN

from sdmetrics.reports.single_table import QualityReport, DiagnosticReport
from sdmetrics.single_table import DCRBaselineProtection

import gdown
import tqdm

# 2. Config

## 2.1 Normal config

In [ ]:
RANDOM_SEEDS = [
    776, 578, 2024, 678
]

OPTUNA_SEEDS     = 42
OPTUNA_N_TRIALS  = 15           # per region
OPTUNA_TIMEOUT   = 3600000         # seconds per region

PHYSICAL_BOUNDS = {
    "GR": (0, 350),      "DT": (30, 200),
    "RHOB_combined": (1.0, 3.5), "RES_DEEP": (0.01, 10_000),
    "HP": (0, 20_000),   "OB": (0, 50_000),
    "DT_NCT": (30, 200), "PPP": (-2_000, 20_000),
}
SENTINEL_VALUES = {-999, -999.25, -999.0}

_WALL_CLOCK, _PEAK_MEM = {}, {}
def _rss_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
def _snap(label, t0, m0):
    _WALL_CLOCK[label] = time.time() - t0
    _PEAK_MEM[label]   = _rss_mb() - m0

GDRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/GAN_Benchmark"

GDRIVE_MOUNTED = False
GDRIVE_RUN_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    GDRIVE_MOUNTED = True
    GDRIVE_RUN_DIR = f"{GDRIVE_CHECKPOINT_DIR}/run_{datetime.now().strftime('%Y%m%d_%H%M')}"
    os.makedirs(GDRIVE_RUN_DIR, exist_ok=True)
    print(f" Google Drive mounted — checkpoints will be saved to {GDRIVE_RUN_DIR}")
except Exception:
    print(" Google Drive not available — checkpoints will be saved locally only.")

## 2.2 Region configurations

In [ ]:
REGION_CONFIGS = {
    "r1": {
        "name": "Region 1 — Missa Keswal, Eastern Potwar Basin",
        "file_ids": ["1ItU6vXk2saTi4ofGo95OHjr39qgsXTan",
                     "1Q0io7g2-K9ALS8enQlFRM1RdI-H1vQQ3",
                     "1Lu-JtVdPckJ2l08fVB8ZMKrdKAMhxLdt"],
        "file_names": ["MISSA-KESWAL-01.csv", "MISSA-KESWAL-02.csv", "MISSA-KESWAL-03.csv"],
        "well_names": ["Well_1", "Well_2", "Well_3"],
        "file_type": "csv",
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP", "HP", "OB", "DT_NCT", "PPP"],
    },
    "r2": {
        "name": "Region 2 — Pindori, Eastern Potwar Basin",
        "file_ids": ["1yvCAJaRbf_rL9wGzE3KJ39j5HaGi1q6g",
                     "1HVFJx2aHu1xf57fgd8WZXDZ79MpPQLY-",
                     "1qOdM_8DLwhJlJc9F6Dn-Z4btoYhgVDH1"],
        "file_names": ["PINDORI-1.csv", "PINDORI-2.csv", "PINDORI-3.csv"],
        "well_names": ["Pindori_1", "Pindori_2", "Pindori_3"],
        "file_type": "csv",
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP", "HP", "OB", "DT_NCT", "PPP"],
    },
    "r3": {
        "name": "Region 3 — Joyamair/Minwal, Eastern Potwar Basin",
        "file_ids": ["1bvH0gDMdd3ZG15HnWXmzFQPqpfjxEspn",
                     "1hjJvfDUEf_LzZFJPohUmzV96UGuPlVPr",
                     "1X8Pj7ukoXDXcRGg3swyuWVFgaBeg4qbb"],
        "file_names": ["JOYAMAIR-4.csv", "MINWAL-2.csv", "MINWAL-X-1.csv"],
        "well_names": ["Joyamair_4", "Minwal_2", "JM_3"],
        "file_type": "csv",
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP", "HP", "OB", "DT_NCT", "PPP"],
    },
    "r4": {
        "name": "Region 4 — IODP Expedition 323, Bering Sea",
        "file_ids": ["1ZNn5wgI6ODe-VV4gZGp-HtUj6mY0bJkP"],
        "file_names": ["merged_logs.xlsx"],
        "well_names": ["U1343E"],
        "file_type": "xlsx",
        "xlsx_slice": (677, 4755),
        "xlsx_col_map": {"DEPTH_m": "DEPTH", "GR_gAPI": "GR", "DT_us_ft": "DT",
                         "RHOB_g_cm3": "RHOB_combined", "RES_DEEP_ohmm": "RES_DEEP"},
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP"],
    },
    "r5": {
        "name": "Region 5 — Volve, North Sea",
        "file_ids": ["1aNypmAzJ3DDrwuGIGcEt0vFrv_mt0QBL8Y6nrmfd-aM",
                     "1kNmd-twgBwAXhg4PHHzFiCEvyeWLg5NApi4Afy5CZeE",
                     "1ljgLkr-yAVPvxMli083cPdYdEEhfLeSbORlFdCFfeGg",
                     "1dRLdtFhT5VjL3ZJvQQRsL5AFhJCgJaVoY0LMBHS2v0M",
                     "13W4pcZpn-hrtM9BBVci_FKVzHQQCLudB_dJ_z4K6-g0"],
        "file_names": ["F-11A.xlsx", "F-11T2.xlsx", "F-12.xlsx", "F-1A.xlsx", "F-1B.xlsx"],
        "well_names": ["F-11A", "F-11T2", "F-12", "F-1A", "F-1B"],
        "file_type": "volve",
        "log_cols": ["GR", "DT", "RHOB_combined", "RES_DEEP"],
    },
}

In [ ]:
def load_region_data(cfg):
    """Download, read, and clean real well data for one region config."""
    for fid in cfg["file_ids"]:
        gdown.download(f"https://drive.google.com/uc?id={fid}", quiet=False)

    if cfg["file_type"] == "csv":
        dfs = []
        for fp in cfg["file_names"]:
            df = pd.read_csv(fp)
            col_map = {c: "RHOB_combined" for c in df.columns
                       if c.upper() == "RHOB_COMBINED" and c != "RHOB_combined"}
            if col_map:
                df = df.rename(columns=col_map)
            dfs.append(df)
        return [clean_well_data(df, label=f"{n} REAL")
                for n, df in zip(cfg["well_names"], dfs)]

    if cfg["file_type"] == "xlsx":
        df = pd.read_excel(cfg["file_names"][0])
        if "xlsx_slice" in cfg:
            df = df.iloc[cfg["xlsx_slice"][0]:cfg["xlsx_slice"][1]]
        if "xlsx_col_map" in cfg:
            df = df.rename(columns=cfg["xlsx_col_map"])
        return [df.dropna().reset_index(drop=True)]

    if cfg["file_type"] == "volve":
        import polars as pl
        dfs = []
        for fp in cfg["file_names"]:
            df_pd = pl.read_excel(fp).drop_nulls().to_pandas()
            rmap = {}
            for c in df_pd.columns:
                cu = c.upper().strip()
                if cu in ("RHOB", "RHOB_COMBINED", "RHOZ"): rmap[c] = "RHOB_combined"
                elif cu in ("RES_DEEP", "RDEP", "RT", "ILD", "RDEEP", "LLD"): rmap[c] = "RES_DEEP"
                elif cu == "GR": rmap[c] = "GR"
                elif cu in ("DT", "DTC", "DTCO"): rmap[c] = "DT"
                elif cu in ("DEPTH", "DEPT"): rmap[c] = "DEPTH"
            df_pd = df_pd.rename(columns=rmap)
            for c in df_pd.columns:
                df_pd[c] = pd.to_numeric(df_pd[c], errors="coerce")
            dfs.append(df_pd.dropna().reset_index(drop=True))
        return dfs

    raise ValueError(f"Unknown file_type: {cfg['file_type']}")

# 3. Data helpers

In [ ]:
def clean_well_data(df, outlier_std=5, verbose=True, label=""):
    df = df.copy()
    n_before = len(df)
    if "SPHI" in df.columns:
        df.drop(columns=["SPHI"], inplace=True)
    log_cols = [c for c in df.columns if c != "DEPTH"]
    for col in log_cols:
        df.loc[df[col].isin(SENTINEL_VALUES), col] = np.nan
        if col in PHYSICAL_BOUNDS:
            lo, hi = PHYSICAL_BOUNDS[col]
            df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
        vals = df[col].dropna()
        if len(vals) >= 10 and vals.std() > 0:
            mu, sig = vals.mean(), vals.std()
            df.loc[(df[col] - mu).abs() > outlier_std * sig, col] = np.nan
    df.dropna(subset=log_cols, how="all", inplace=True)
    df.reset_index(drop=True, inplace=True)
    if verbose:
        removed = n_before - len(df)
        pct = 100 * removed / n_before if n_before else 0
        print(f"  [{label}]  {n_before} → {len(df)} rows  (removed {removed}, {pct:.1f}%)")
    return df

In [ ]:
def compare(df_real, df_syn, bins=50):
    """Per-column JSD + Wasserstein (absolute + normalised)."""
    rows = []
    for col in df_syn.columns:
        if col == "DEPTH":
            continue
        s1 = df_real[col].dropna().values
        s2 = df_syn[col].dropna().values
        if len(s1) < 2 or len(s2) < 2:
            continue
        wd = wasserstein_distance(s1, s2)
        pooled = np.sqrt((s1.var() + s2.var()) / 2)
        wd_norm = wd / pooled if pooled > 0 else 0.0

        lo = min(s1.min(), s2.min())
        hi = max(s1.max(), s2.max())
        edges = np.linspace(lo, hi, bins)
        h1, _ = np.histogram(s1, bins=edges, density=True)
        h2, _ = np.histogram(s2, bins=edges, density=True)
        h1 = h1 / h1.sum() if h1.sum() > 0 else h1
        h2 = h2 / h2.sum() if h2.sum() > 0 else h2
        rows.append({
            "column": col, "js_divergence": jensenshannon(h1, h2),
            "wasserstein_dist": wd, "wasserstein_norm": wd_norm,
        })
    return pd.DataFrame(rows).set_index("column")

# 4. CTGAN + Optuna

In [ ]:
def _parse_dim(s_or_tuple):
    """Optuna gives us 'a,b,c' strings; CTGAN wants (a, b, c) tuples."""
    if isinstance(s_or_tuple, str):
        return tuple(int(x) for x in s_or_tuple.split(","))
    return tuple(s_or_tuple)

def ctgan_objective(train_df, log_cols):
    """Build an Optuna objective that trains CTGAN and returns mean JSD."""
    def objective(trial):
        params = {
            "embedding_dim":     trial.suggest_categorical("embedding_dim", [128, 256]),
            "generator_dim":     trial.suggest_categorical("generator_dim", ["128,128", "256,256"]),
            "lr":                trial.suggest_float("lr", 1e-4, 5e-4, log=True),
            "batch_size":        trial.suggest_categorical("batch_size", [500, 1000]),
            "epochs":            trial.suggest_categorical("epochs", [100, 200, 300, 400, 500, 600]),
        }

        # Translate search-space params into CTGAN's actual init arguments
        fit_params = dict(params)
        fit_params["generator_dim"]     = _parse_dim(params["generator_dim"])
        fit_params["discriminator_dim"] = fit_params["generator_dim"]  # tied to generator_dim
        lr = fit_params.pop("lr")
        fit_params["generator_lr"]     = lr
        fit_params["discriminator_lr"] = lr
        fit_params["pac"]                 = 10
        fit_params["discriminator_steps"] = 1

        scores = []
        for seed in OPTUNA_SEEDS:
            torch.manual_seed(seed); np.random.seed(seed)
            try:
                model = CTGAN(**fit_params, verbose=False,
                              enable_gpu=torch.cuda.is_available())
                model.fit(train_df[log_cols])
                syn = model.sample(len(train_df))
                js = compare(train_df[log_cols], syn)["js_divergence"].mean()
                scores.append(js)
                del model, syn
            except Exception as e:
                print(f"  Trial failed: {e}")
                return float("inf")

        return float(np.mean(scores))
    return objective

def build_synthetic_ctgan(seed, params, n_rows, train_df, log_cols, return_model=False):
    """Train CTGAN with one seed and sample n_rows rows. Drop-in for 20-seed loop.

    If return_model=True, returns (synthetic_df, fitted_model) so callers can
    checkpoint the trained weights.
    """
    torch.manual_seed(seed); np.random.seed(seed)
    fit_params = dict(params)
    fit_params["generator_dim"]     = _parse_dim(params["generator_dim"])

    fit_params["discriminator_dim"] = fit_params["generator_dim"]  # tied to generator_dim
    # Expand the tied learning rate into CTGAN's two separate arguments
    if "lr" in fit_params:
        lr = fit_params.pop("lr")
        fit_params["generator_lr"]     = lr
        fit_params["discriminator_lr"] = lr
    model = CTGAN(**fit_params, verbose=False, enable_gpu=torch.cuda.is_available())
    model.fit(train_df[log_cols])
    syn = model.sample(n_rows)
    if return_model:
        return syn, model
    return syn

def sliced_wasserstein_distance(X, Y, n_projections=1000, seed=42):
    n = min(len(X), len(Y))
    rng = np.random.default_rng(seed)
    X = X[rng.choice(len(X), n, replace=False)]
    Y = Y[rng.choice(len(Y), n, replace=False)]
    dists = []
    for _ in range(n_projections):
        d = rng.standard_normal(X.shape[1]); d /= np.linalg.norm(d)
        dists.append(np.mean(np.abs(np.sort(X @ d) - np.sort(Y @ d))))
    return float(np.mean(dists))

def mmd_rbf(X, Y, gamma=None):
    from scipy.spatial.distance import cdist
    if gamma is None:
        dists = cdist(X, Y, "sqeuclidean")
        gamma = 1.0 / np.median(dists[dists > 0])
    XX = np.exp(-gamma * cdist(X, X, "sqeuclidean"))
    YY = np.exp(-gamma * cdist(Y, Y, "sqeuclidean"))
    XY = np.exp(-gamma * cdist(X, Y, "sqeuclidean"))
    return float(XX.mean() + YY.mean() - 2 * XY.mean())

def energy_distance(X, Y):
    from scipy.spatial.distance import cdist
    return float(2 * cdist(X, Y).mean() - cdist(X, X).mean() - cdist(Y, Y).mean())

def mmd_permutation_test(X, Y, n_permutations=500, seed=42):
    """Permutation test for MMD significance. Returns (observed_mmd, p_value)."""
    rng = np.random.default_rng(seed)
    combined = np.vstack([X, Y])
    n = len(X)
    observed = mmd_rbf(X, Y)
    count = 0
    for _ in range(n_permutations):
        p = rng.permutation(len(combined))
        if mmd_rbf(combined[p[:n]], combined[p[n:]]) >= observed:
            count += 1
    return float(observed), float(count / n_permutations)

def lag_autocorrelation(series, max_lag=50):
    x = np.asarray(series, dtype=float); x = x[~np.isnan(x)]
    if len(x) <= max_lag + 1 or x.std() == 0:
        return np.zeros(max_lag)
    x = (x - x.mean()) / x.std()
    return np.array([np.corrcoef(x[:len(x)-lag], x[lag:])[0, 1]
                     for lag in range(1, max_lag + 1)])

def experimental_variogram(values, max_lag=50):
    lags = range(1, max_lag + 1)
    return (np.array(list(lags)),
            np.array([0.5 * np.mean((values[h:] - values[:-h]) ** 2) for h in lags]))

def autocorrelation_length(acf):
    below = np.where(acf < 1.0 / np.e)[0]
    return int(below[0] + 1) if len(below) > 0 else len(acf)

In [ ]:
def full_evaluation(real_df, synth_df, log_cols):
    """All extended metrics. Returns dict of tables + scalar dicts."""
    from scipy import stats as sstats
    from sklearn.preprocessing import StandardScaler

    # Fidelity (JSD, WD, KS, Anderson-Darling per column)
    fid_rows = []
    for col in log_cols:
        r = real_df[col].dropna().values
        s = synth_df[col].dropna().values
        lo, hi = min(r.min(), s.min()), max(r.max(), s.max())
        bins = np.linspace(lo, hi, 51)
        p = np.histogram(r, bins=bins, density=True)[0] + 1e-12
        q = np.histogram(s, bins=bins, density=True)[0] + 1e-12
        p /= p.sum(); q /= q.sum()
        m = 0.5 * (p + q)
        jsd = 0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m))
        wd = sstats.wasserstein_distance(r, s)
        ks_stat, ks_p = sstats.ks_2samp(r, s)
        try:
            ad_stat, _, ad_p = sstats.anderson_ksamp([r, s])
        except Exception:
            ad_stat, ad_p = float("nan"), float("nan")
        fid_rows.append({"Log": col, "JSD": round(jsd, 4), "WD": round(wd, 2),
                         "KS_stat": round(ks_stat, 4), "KS_p": round(ks_p, 4),
                         "AD_stat": round(ad_stat, 4), "AD_p": round(ad_p, 4)})
    fidelity = pd.DataFrame(fid_rows)

    # Correlation structure (Frobenius distance + pairwise)
    frob = float(np.linalg.norm(real_df[log_cols].corr().values
                                - synth_df[log_cols].corr().values, "fro"))
    corr_rows = []
    for c1, c2 in combinations(log_cols, 2):
        rr = real_df[c1].corr(real_df[c2])
        rs = synth_df[c1].corr(synth_df[c2])
        corr_rows.append({"Pair": f"{c1}–{c2}", "Corr_Real": round(rr, 4),
                          "Corr_Synth": round(rs, 4),
                          "Abs_Error": round(abs(rr - rs), 4)})
    correlation = pd.DataFrame(corr_rows)

    # Multivariate distances (standardised, subsampled for speed)
    scaler = StandardScaler().fit(real_df[log_cols])
    Xr = scaler.transform(real_df[log_cols])
    Xs = scaler.transform(synth_df[log_cols])
    MAX_N = 2000
    rng = np.random.RandomState(42)
    if len(Xr) > MAX_N: Xr = Xr[rng.choice(len(Xr), MAX_N, replace=False)]
    if len(Xs) > MAX_N: Xs = Xs[rng.choice(len(Xs), MAX_N, replace=False)]
    multivariate = {
        "swd":             sliced_wasserstein_distance(Xr, Xs),
        "mmd_rbf":         mmd_rbf(Xr, Xs),
        "energy_distance": energy_distance(Xr, Xs),
        "frobenius_corr":  frob,
    }
    # MMD permutation significance (v21 parity)
    try:
        mmd_obs, mmd_p = mmd_permutation_test(Xr, Xs)
        multivariate["mmd_obs"] = mmd_obs
        multivariate["mmd_p"]   = mmd_p
    except Exception as e:
        print(f"  MMD permutation test failed: {e}")

    # Spatial continuity (ACF + variogram RMSE per column)
    spatial_rows = []
    for col in log_cols:
        ar  = lag_autocorrelation(real_df[col].values)
        asy = lag_autocorrelation(synth_df[col].values)
        _, gr = experimental_variogram(real_df[col].values)
        _, gs = experimental_variogram(synth_df[col].values)
        spatial_rows.append({
            "Log": col,
            "ACL_Real":       autocorrelation_length(ar),
            "ACL_Synth":      autocorrelation_length(asy),
            "ACF_RMSE":       round(float(np.sqrt(np.mean((ar - asy) ** 2))), 4),
            "Sill_Real":      round(float(gr[-1]), 4),
            "Sill_Synth":     round(float(gs[-1]), 4),
            "Variogram_RMSE": round(float(np.sqrt(np.mean((gr - gs) ** 2))), 4),
        })
    spatial = pd.DataFrame(spatial_rows)

    # SDMetrics (Quality, Diagnostic, DCR privacy baseline + detailed breakdowns)
    sdm = {}
    sdm_shapes = None
    sdm_trends = None
    dcr_breakdown = None
    try:
        from sdmetrics.reports.single_table import QualityReport, DiagnosticReport
        from sdmetrics.single_table import DCRBaselineProtection
        meta = {"columns": {c: {"sdtype": "numerical"} for c in log_cols}}
        ql = QualityReport();    ql.generate(real_df[log_cols], synth_df[log_cols], meta)
        dg = DiagnosticReport(); dg.generate(real_df[log_cols], synth_df[log_cols], meta)
        dcr = DCRBaselineProtection.compute(
            real_data=real_df[log_cols], synthetic_data=synth_df[log_cols], metadata=meta
        )
        sdm = {"quality_score":    float(ql.get_score()),
               "diagnostic_score": float(dg.get_score()),
               "dcr_score":        float(dcr)}
        # Detailed breakdowns (v21 parity)
        try:
            sdm_shapes = ql.get_details(property_name="Column Shapes")
            sdm_trends = ql.get_details(property_name="Column Pair Trends")
        except Exception as e:
            print(f"  SDMetrics get_details failed: {e}")
        try:
            dcr_breakdown = DCRBaselineProtection.compute_breakdown(
                real_data=real_df[log_cols], synthetic_data=synth_df[log_cols], metadata=meta
            )
        except Exception as e:
            print(f"  DCR breakdown failed: {e}")
    except Exception as e:
        print(f"  SDMetrics failed: {e}")

    return {"fidelity": fidelity, "correlation": correlation,
            "multivariate": multivariate, "spatial": spatial, "sdmetrics": sdm,
            "sdmetrics_shapes": sdm_shapes, "sdmetrics_trends": sdm_trends,
            "dcr_breakdown": dcr_breakdown}

# 5. Main per-region pipeline

In [ ]:
import os, shutil
from datetime import datetime

def load_checkpoint(tag):
    """Return best_params dict if a checkpoint exists, else None."""
    hp_path = f"checkpoints/{tag}/best_params_{tag}.json"
    if os.path.exists(hp_path):
        with open(hp_path) as f:
            bp = json.load(f)
        print(f"  Loaded checkpoint for {tag} from {hp_path}")
        return bp
    return None

In [ ]:
def save_checkpoint(cfg, tag, best_params, study, ev, table3,
                    synth_model, wall_time):
    """Organize everything under checkpoints/{tag}/, save .pt weights,
    Optuna trials, a markdown summary, then mirror to Google Drive."""
    local_dir = f"checkpoints/{tag}"
    os.makedirs(local_dir, exist_ok=True)

    hp_path = f"{local_dir}/best_params_{tag}.json"
    with open(hp_path, "w") as f:
        json.dump(best_params, f, indent=2)

    pt_path = f"{local_dir}/ctgan_{tag}.pkl"
    synth_model.save(pt_path)

    trials_path = f"{local_dir}/trials_{tag}.csv"
    try:
        study.trials_dataframe().to_csv(trials_path, index=False)
    except Exception as e:
        print(f"  ⚠ Could not save trials CSV: {e}")
        trials_path = None

    for src in [f"table3_{tag}.csv",
                f"table3_paper_{tag}.csv",
                f"fidelity_{tag}.csv",
                f"correlation_{tag}.csv",
                f"spatial_{tag}.csv",
                f"multivariate_{tag}.json",
                f"sdmetrics_{tag}.json",
                f"sdmetrics_shapes_{tag}.csv",
                f"sdmetrics_trends_{tag}.csv",
                f"dcr_breakdown_{tag}.csv",
                f"dcr_breakdown_{tag}.json",
                f"baseline_real_vs_real_{tag}.csv"]:
        if os.path.exists(src):
            shutil.copy2(src, local_dir)

    md_path = f"{local_dir}/summary_{tag}.md"
    with open(md_path, "w") as f:
        f.write(f"# CTGAN Checkpoint — {cfg['name']}\n\n")
        f.write(f"- **Tag:** `{tag}`\n")
        f.write(f"- **Timestamp:** {datetime.now().isoformat(timespec='seconds')}\n")
        f.write(f"- **Wall time:** {wall_time/60:.1f} min\n")
        f.write(f"- **Best JSD (Optuna):** {study.best_value:.4f}\n")
        f.write(f"- **Trials:** {len(study.trials)}\n\n")
        f.write(f"## Best hyperparameters\n```json\n")
        f.write(json.dumps(best_params, indent=2))
        f.write("\n```\n\n## Final benchmark (Table 3)\n")
        f.write(table3.to_markdown(floatfmt=".4f"))
        f.write("\n\n## Multivariate metrics\n```json\n")
        f.write(json.dumps(ev["multivariate"], indent=2))
        f.write("\n```\n")
        if ev.get("sdmetrics"):
            f.write("\n## SDMetrics\n```json\n")
            f.write(json.dumps(ev["sdmetrics"], indent=2))
            f.write("\n```\n")

    print(f"\n  Checkpoint saved locally: {local_dir}/")
    print(f"     ├─ best_params_{tag}.json       (hyperparameters)")
    print(f"     ├─ ctgan_{tag}.pt               ⭐ (model weights)")
    if trials_path: print(f"     ├─ trials_{tag}.csv             (Optuna trials)")
    print(f"     ├─ summary_{tag}.md")
    print(f"     ├─ table3_{tag}.csv, table3_paper_{tag}.csv")
    print(f"     ├─ fidelity, correlation, spatial, multivariate, sdmetrics")
    print(f"     ├─ sdmetrics_shapes, sdmetrics_trends, dcr_breakdown")
    print(f"     └─ baseline_real_vs_real_{tag}.csv")

    if GDRIVE_MOUNTED and GDRIVE_RUN_DIR:
        try:
            gdrive_dir = f"{GDRIVE_RUN_DIR}/{tag}"
            os.makedirs(gdrive_dir, exist_ok=True)
            for fname in os.listdir(local_dir):
                shutil.copy2(f"{local_dir}/{fname}", gdrive_dir)
            print(f"  Also saved to Google Drive: {gdrive_dir}/")
        except Exception as e:
            print(f"  ⚠ Google Drive save failed: {e}")
            print(f"     Local checkpoint is safe at: {local_dir}/")

In [ ]:
def run_region(cfg, tag):
    print(f"\n{'=' * 70}\n  {cfg['name']}  (tag={tag})\n{'=' * 70}")
    region_t0 = time.time()

    # 6.1 Load + clean
    real_dfs = load_region_data(cfg)
    log_cols = cfg["log_cols"]
    train_df = pd.concat([df[log_cols] for df in real_dfs], ignore_index=True).dropna()
    print(f"  Training rows (pooled across {len(real_dfs)} wells): {len(train_df):,}")

    # 6.2 Baseline pairwise real-vs-real (only meaningful with ≥2 wells)
    baseline = None
    if len(real_dfs) >= 2:
        pair_comps = [compare(a[log_cols], b[log_cols])
                      for (_, a), (_, b) in combinations(zip(cfg["well_names"], real_dfs), 2)]
        baseline = pd.concat(pair_comps).groupby(level=0).mean()
        print(f"\nBaseline (pairwise real-vs-real, {len(pair_comps)} pairs):")
        print(baseline[["js_divergence", "wasserstein_norm"]].to_string(float_format="%.4f"))
        baseline.to_csv(f"baseline_real_vs_real_{tag}.csv")

    # 6.3 Optuna hyperparameter search
    print(f"\nOptuna search: {OPTUNA_N_TRIALS} trials × {len(OPTUNA_SEEDS)} seeds each...")
    t0, m0 = time.time(), _rss_mb()
    study = optuna.create_study(
        study_name=f"{tag}_ctgan",
        direction="minimize",
        sampler=TPESampler(seed=42),
    )
    study.optimize(ctgan_objective(train_df, log_cols),
                   n_trials=OPTUNA_N_TRIALS,
                   timeout=OPTUNA_TIMEOUT,
                   show_progress_bar=True)
    _snap(f"{tag}_optuna", t0, m0)

    BEST = dict(study.best_params)
    print(f"\nBest CTGAN params — {tag}  "
          f"(JS={study.best_value:.4f}, {(time.time()-t0)/60:.1f} min):")
    for k, v in BEST.items():
        print(f"  {k:25s} = {v}")

    # 6.4 Final evaluation across all seeds
    print(f"\nFinal evaluation across {len(RANDOM_SEEDS)} seeds...")
    t0, m0 = time.time(), _rss_mb()
    per_well = {name: [] for name in cfg["well_names"]}
    for seed in tqdm.tqdm(RANDOM_SEEDS, desc=f"  {tag}"):
        for name, df_real in zip(cfg["well_names"], real_dfs):
            n_rows = len(df_real)
            df_syn = build_synthetic_ctgan(seed, BEST, n_rows, train_df, log_cols)
            per_well[name].append(compare(df_real[log_cols], df_syn))
    _snap(f"{tag}_final_bench", t0, m0)

    # 6.5 Table 3 (+ paper-ready flat version)
    avg = {n: pd.concat(v).groupby(level=0).mean() for n, v in per_well.items()}
    std = {n: pd.concat(v).groupby(level=0).std()  for n, v in per_well.items()}
    grand      = pd.concat(avg.values()).groupby(level=0).mean()
    grand_std  = pd.concat(std.values()).groupby(level=0).mean()
    n          = len(RANDOM_SEEDS)

    table3 = pd.DataFrame({
        "js_mean": grand["js_divergence"],
        "js_std":  grand_std["js_divergence"],
        "js_ci95": 1.96 * grand_std["js_divergence"] / np.sqrt(n),
        "wd_mean": grand["wasserstein_norm"],
        "wd_std":  grand_std["wasserstein_norm"],
        "wd_ci95": 1.96 * grand_std["wasserstein_norm"] / np.sqrt(n),
    })
    table3.to_csv(f"table3_{tag}.csv")

    # Paper-ready flat Table 3 (v21 parity)
    paper_rows = []
    for col in table3.index:
        for metric_prefix, label in [("js", "JSD"), ("wd", "WD_norm")]:
            mean = table3.loc[col, f"{metric_prefix}_mean"]
            std_ = table3.loc[col, f"{metric_prefix}_std"]
            ci95 = table3.loc[col, f"{metric_prefix}_ci95"]
            paper_rows.append({
                "Log": col, "Metric": label,
                "Mean": round(float(mean), 4), "Std": round(float(std_), 4),
                "CI_95_lo": round(float(mean - ci95), 4),
                "CI_95_hi": round(float(mean + ci95), 4),
                "Display": f"{mean:.4f} ± {std_:.4f}",
            })
    table3_paper = pd.DataFrame(paper_rows)
    table3_paper.to_csv(f"table3_paper_{tag}.csv", index=False)

    with open(f"best_params_{tag}.json", "w") as f:
        json.dump(BEST, f, indent=2)

    print(f"\n[Table 3 — {tag}]")
    print(table3.to_string(float_format="%.4f"))
    print(f"  Saved: table3_{tag}.csv, table3_paper_{tag}.csv, best_params_{tag}.json")

    # 6.6 Extended evaluation (one representative synthetic dataset, seed=42)
    print(f"\nExtended evaluation (seed=42 synthetic)...")
    real_data  = pd.concat([df[log_cols] for df in real_dfs], ignore_index=True).dropna()
    synth_data, synth_model = build_synthetic_ctgan(
        42, BEST, len(real_data), train_df, log_cols, return_model=True
    )
    ev = full_evaluation(real_data, synth_data, log_cols)

    ev["fidelity"].to_csv(f"fidelity_{tag}.csv", index=False)
    ev["correlation"].to_csv(f"correlation_{tag}.csv", index=False)
    ev["spatial"].to_csv(f"spatial_{tag}.csv", index=False)
    with open(f"multivariate_{tag}.json", "w") as f:
        json.dump(ev["multivariate"], f, indent=2)
    with open(f"sdmetrics_{tag}.json", "w") as f:
        json.dump(ev["sdmetrics"], f, indent=2)

    # Detailed SDMetrics breakdowns (v21 parity)
    if ev.get("sdmetrics_shapes") is not None:
        try:
            ev["sdmetrics_shapes"].to_csv(f"sdmetrics_shapes_{tag}.csv", index=False)
        except Exception as e:
            print(f"  Could not save sdmetrics_shapes: {e}")
    if ev.get("sdmetrics_trends") is not None:
        try:
            ev["sdmetrics_trends"].to_csv(f"sdmetrics_trends_{tag}.csv", index=False)
        except Exception as e:
            print(f"  Could not save sdmetrics_trends: {e}")
    if ev.get("dcr_breakdown") is not None:
        try:
            dcr_b = ev["dcr_breakdown"]
            if isinstance(dcr_b, pd.DataFrame):
                dcr_b.to_csv(f"dcr_breakdown_{tag}.csv", index=False)
            else:
                with open(f"dcr_breakdown_{tag}.json", "w") as f:
                    json.dump(dcr_b, f, indent=2, default=str)
        except Exception as e:
            print(f"  Could not save dcr_breakdown: {e}")

    print(f"\n[Fidelity — {tag}]")
    print(ev["fidelity"].to_string(index=False))
    print(f"\n[Correlation — {tag}]   (Frobenius = {ev['multivariate']['frobenius_corr']:.4f})")
    print(ev["correlation"].to_string(index=False))
    print(f"\n[Spatial — {tag}]")
    print(ev["spatial"].to_string(index=False))
    print(f"\n[Multivariate — {tag}]")
    for k, v in ev["multivariate"].items():
        print(f"  {k:20s} = {v:.6f}")
    if ev["sdmetrics"]:
        print(f"\n[SDMetrics — {tag}]")
        for k, v in ev["sdmetrics"].items():
            print(f"  {k:20s} = {v:.4f}")

    wall_time = time.time() - region_t0
    print(f"\n  Region wall time: {wall_time/60:.1f} min")

    # 6.7 Checkpoint: HPs + .pt weights + trials + summary → local + Drive
    save_checkpoint(cfg, tag, BEST, study, ev, table3, synth_model, wall_time)

    del real_dfs, train_df, per_well, study, real_data, synth_data, synth_model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return table3, BEST

# 6. Run all

In [ ]:
def run_all():
    t0 = time.time()
    results = {}
    for tag in ["r1", "r2", "r3", "r4", "r5"]:
        results[tag] = run_region(REGION_CONFIGS[tag], tag)
    print(f"\n{'=' * 70}\n  ALL REGIONS DONE in {(time.time()-t0)/60:.1f} min\n{'=' * 70}")
    return results

run_all()